# InternScenes → USD / Isaac Sim Pipeline

This notebook walks the InternScenes → USD (Isaac Sim) pipeline end to end, **executing** each stage and verifying that its output file is actually produced — not just pasting commands.

## Stages

1. **Environment check** — confirm the project root, Python venv, Blender, and inputs exist
2. **GLB → USD (single scene)** — `scripts/glb_to_usd.py`
3. **GLB → USD (batch of 5)** — `scripts/usd_all.sh`
4. **GLB → perspective PNG** — Blender headless (EEVEE), `scripts/glb_render.py`
5. **2D top-down projection** — `scripts/topdown_projection.py`
6. **Scene inventory** — scan every InternScenes scene across the 4 datasets

## How to read this notebook

Each stage is introduced by a Markdown cell that explains the **principle**, followed by a code cell that **runs the real command** and **checks the output file**. Re-running a cell reproduces the result.

> **Tip:** rendering and exporting *every* sampled scene (Blender view + top-down + scene-info JSON) is handled by `scripts/batch_pipeline.py`; see the project **README**. This notebook focuses on the single-scene / small-batch commands.


## 0. Environment check

Before running any command, confirm the prerequisites are in place:

- **Project root**: `/home/ubadmin/projects/InternScenes2isaacsim`
- **Python venv** `.venv311` — has `trimesh`, `pxr`/OpenUSD, `open3d`, `matplotlib`; used by `glb_to_usd.py` and `topdown_projection.py`
- **Blender binary** `/home/ubadmin/tools/blender/blender-5.1.2-linux-x64/blender` — headless renderer used by `glb_render.py`
- **Composed GLB scenes** `output/composed/<dataset>/<scene>/glb_scene.glb` — the shared input for USD conversion and rendering

The cell below checks each path; any `MISSING` entry means the corresponding command cannot run yet.


In [ ]:
import os

PROJ = "/home/ubadmin/projects/InternScenes2isaacsim"
BLENDER = "/home/ubadmin/tools/blender/blender-5.1.2-linux-x64/blender"
VENV_PY = os.path.join(PROJ, ".venv311/bin/python")

checks = {
    "project root": PROJ,
    "venv311 python": VENV_PY,
    "blender binary": BLENDER,
    "scripts/glb_to_usd.py": os.path.join(PROJ, "scripts/glb_to_usd.py"),
    "scripts/usd_all.sh": os.path.join(PROJ, "scripts/usd_all.sh"),
    "scripts/glb_render.py": os.path.join(PROJ, "scripts/glb_render.py"),
    "scripts/topdown_projection.py": os.path.join(PROJ, "scripts/topdown_projection.py"),
    "scene0001_00 glb": os.path.join(PROJ, "output/composed/scannet/scene0001_00/glb_scene.glb"),
}

for label, path in checks.items():
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"[{status}] {label}: {path}")


## 1. GLB → USD (single scene)```bashcd /home/ubadmin/projects/InternScenes2isaacsim.venv311/bin/python scripts/glb_to_usd.py \  --glb output/composed/scannet/scene0001_00/glb_scene.glb \  --out output/normalized/scannet_scene0001_00/scene.usd```**Principle**1. `trimesh.load(glb)` parses the GLB scene into a trimesh scene graph (one geometry + transform matrix per object).2. A new USD Stage is created with `pxr.Usd.Stage`, and each object is built under `/Objects/xform_*/mesh`:   - `Mesh` — writes vertices, faces, and other topology data   - `Xform` — writes the object's world-space transform (position, rotation, scale)   - `Shader` — writes BaseColor so the GLB material colour carries into the USD3. Finally `stage.Export(usd)` writes the whole Stage to a `.usd` file.**Output:** `output/normalized/<ds>_<id>/scene.usd` (default normalized path; `--out` still works) — contains materials and world transforms, openable directly by Isaac Sim / usdview.The cell below **runs this command for real** and checks that the output file was produced.

In [ ]:
import subprocess, os

glb = "output/composed/scannet/scene0001_00/glb_scene.glb"
usd_out = "output/normalized/scannet_scene0001_00/scene.usd"

cmd = [VENV_PY, "scripts/glb_to_usd.py", "--glb", glb]  # normalized default
print("running:", " ".join(cmd))

proc = subprocess.run(cmd, cwd=PROJ, capture_output=True, text=True, timeout=120)
print("--- stdout ---")
print(proc.stdout.strip())
if proc.returncode != 0:
    print("--- stderr ---")
    print(proc.stderr.strip())
print("exit_code:", proc.returncode)

usd_path = os.path.join(PROJ, usd_out)
if os.path.exists(usd_path):
    size_kb = os.path.getsize(usd_path) / 1024
    print(f"\nOK  generated: {usd_path} ({size_kb:.1f} KB)")
else:
    print(f"\nFAIL not found: {usd_path}")


## 2. GLB → USD (batch of 5)

```bash
cd /home/ubadmin/projects/InternScenes2isaacsim
bash scripts/usd_all.sh
```

**Principle:** `scripts/usd_all.sh` is a single `for` loop over a fixed set of 5 scenes (3rscan / arkitscenes / scannet ×2 / matterport3d). For each it calls the same `glb_to_usd.py` from section 1 — i.e. the same command run 5 times:

```bash
for scene in "${SCENES[@]}"; do
  .venv311/bin/python scripts/glb_to_usd.py --glb="output/composed/$scene/glb_scene.glb"   # writes to output/normalized/<ds>_<id>/scene.usd
done
```

**Output:** `output/normalized/<ds>_<id>/scene.usd` (default normalized path; `--out` still works) × 5.

The cell below **runs the script for real** (roughly 10 s per scene, 5 scenes in well under a minute) and checks each USD was produced.


In [ ]:
import subprocess, os, time

SCENES = [
    "3rscan/095821fb-e2c2-2de1-94df-20f2cb423bcb",
    "arkitscenes/Training/43896449",
    "scannet/scene0001_00",
    "scannet/scene0000_00",
    "matterport3d/B6ByNegPMKs/region51",
]

t0 = time.time()
proc = subprocess.run(["bash", "scripts/usd_all.sh"], cwd=PROJ, capture_output=True, text=True, timeout=600)
elapsed = time.time() - t0

print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("--- stderr ---")
    print(proc.stderr[-2000:])
print(f"exit_code: {proc.returncode}, elapsed: {elapsed:.1f}s")

print("\nper-scene check:")
for scene in SCENES:
    p = os.path.join(PROJ, "output/normalized", scene.replace("/", "_"), "scene.usd")
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {scene}")


## 3. Render path ①: Blender GLB → perspective PNG

```bash
cd /home/ubadmin/projects/InternScenes2isaacsim
/home/ubadmin/tools/blender/blender-5.1.2-linux-x64/blender --background \
  --python scripts/glb_render.py -- \
  --glb output/composed/scannet/scene0001_00/glb_scene.glb \
  --out output/render/scannet/scene0001_00 --engine=EEVEE
```

**Principle**

- `--background`: start Blender headless — ideal for batch rendering on a server.
- `--python scripts/glb_render.py`: run the script, which internally:
  1. imports the `--glb` scene using Blender's built-in glTF importer
  2. places a camera *inside* the room (a first-person / interior view)
  3. renders a perspective image with the `--engine` engine (here EEVEE, the fast real-time engine whose materials are good enough)
  4. saves the result as `<out>/perspective.png`

This is the **most reliable** render path: it depends only on the local Blender and needs no extra GPU cluster or remote environment.

The cell below **runs this command for real** (roughly 8–10 s per scene) and shows the result.


In [ ]:
import subprocess, os

cmd = [
    BLENDER, "--background", "--python", "scripts/glb_render.py", "--",
    "--glb", "output/composed/scannet/scene0001_00/glb_scene.glb",
    "--out", "output/render/scannet/scene0001_00", "--engine=EEVEE",
]
print("running:", " ".join(cmd))

proc = subprocess.run(cmd, cwd=PROJ, capture_output=True, text=True, timeout=300)
# Blender's log is long; keep only the lines relevant to the render result
for line in proc.stdout.splitlines():
    if any(k in line for k in ("render saved", "rendering", "DONE", "Error", "Saved")):
        print(line)
print("exit_code:", proc.returncode)

png = os.path.join(PROJ, "output/render/scannet/scene0001_00/perspective.png")
if os.path.exists(png):
    print(f"\nOK  rendered: {png} ({os.path.getsize(png)/1024:.1f} KB)")
else:
    print(f"\nFAIL no render result: {png}")


In [ ]:
from IPython.display import Image, display

png = os.path.join(PROJ, "output/render/scannet/scene0001_00/perspective.png")
if os.path.exists(png):
    display(Image(filename=png, width=480))
else:
    print("render file not found — cannot preview.")


## 4. Render path ②: 2D top-down projection

```bash
cd /home/ubadmin/projects/InternScenes2isaacsim
.venv311/bin/python scripts/topdown_projection.py scannet/scene0001_00
```

**Principle**

- This path does **not** use Blender. It reads the scene's layout (the objects' 3D bounding boxes) and projects every object onto the ground plane (a top-down view).
- The image shows:
  - the Go2 robot's position (the script computes a walkable spot)
  - the clearance from the robot to the nearest obstacle (metres)
  - the ground-plane footprints of every obstacle
  - a metre-scale coordinate grid (to gauge real-world dimensions)
- Compared with a Blender perspective render, this top-down view is better for **quickly checking that the layout is sensible** (e.g. is the Go2 stuck in a wall, is the corridor wide enough) without waiting for the render engine.

**Output:** `output/topdown/<scene>_topdown.png`

The cell below **runs this command for real** (under 1 s) and shows the result.


In [ ]:
import subprocess, os

cmd = [VENV_PY, "scripts/topdown_projection.py", "scannet/scene0001_00"]
print("running:", " ".join(cmd))

proc = subprocess.run(cmd, cwd=PROJ, capture_output=True, text=True, timeout=60)
print(proc.stdout.strip())
if proc.returncode != 0:
    print("--- stderr ---")
    print(proc.stderr.strip())
print("exit_code:", proc.returncode)

topdown_png = os.path.join(PROJ, "output/topdown/scannet_scene0001_00_topdown.png")
if os.path.exists(topdown_png):
    print(f"\nOK  generated: {topdown_png}")
else:
    print(f"\nFAIL not found: {topdown_png}")


In [ ]:
from IPython.display import Image, display

topdown_png = os.path.join(PROJ, "output/topdown/scannet_scene0001_00_topdown.png")
if os.path.exists(topdown_png):
    display(Image(filename=topdown_png, width=480))
else:
    print("top-down file not found — cannot preview.")


## 5. Scene inventory (all scenes, grouped by dataset)

This section lists the IDs of **every InternScenes scene** under `data/Layout_info/`, grouped by data source (the "scene type" from the tables above), so you can see what each dataset's **IDs look like**.

**The 4 datasets (scene types) and their ID format:**

| dataset | directory | ID format example | notes |
|---|---|---|---|
| `scannet` | `data/Layout_info/scannet/<id>/` | `scene0001_00` | `scene` + 4-digit scene no. + `_` + 2-digit reconstruction no. |
| `3rscan` | `data/Layout_info/3rscan/<id>/` | `095821fb-e2c2-2de1-94df-20f2cb423bcb` | 32-char UUID (flat, no hierarchy) |
| `arkitscenes` | `data/Layout_info/arkitscenes/<split>/<id>/` | `Training/43896449` | grouped by `Training`/`Validation`, then an 8-digit ID |
| `matterport3d` | `data/Layout_info/matterport3d/<building>/<region>/` | `B6ByNegPMKs/region51` | grouped by a building hash, then `region<N>` (room no.) |

The code below scans `data/Layout_info/` (each folder containing a `layout.json` is a scene), counts the total per dataset, and previews the first 20 IDs of each so you can see the real IDs.


In [ ]:
import os

LAYOUT_DIR = os.path.join(PROJ, "data", "Layout_info")


def scan_dataset(dataset_dir: str) -> list[str]:
    """Walk a dataset dir and return every scene id (folder containing
    layout.json), relative to the dataset root, in POSIX form
    (e.g. 'Training/43896449')."""
    scene_ids = []
    for root, dirs, files in os.walk(dataset_dir):
        if "layout.json" in files:
            rel = os.path.relpath(root, dataset_dir).replace(os.sep, "/")
            scene_ids.append(rel)
    scene_ids.sort()
    return scene_ids


DATASETS = ["scannet", "3rscan", "arkitscenes", "matterport3d"]
inventory = {}
for name in DATASETS:
    d = os.path.join(LAYOUT_DIR, name)
    inventory[name] = scan_dataset(d) if os.path.isdir(d) else []

total = sum(len(v) for v in inventory.values())
print(f"Total InternScenes scenes: {total}\n")
for name in DATASETS:
    ids = inventory[name]
    print(f"=== {name}  ({len(ids)} scenes) ===")
    for sid in ids[:20]:
        print("  ", sid)
    if len(ids) > 20:
        print(f"   ... {len(ids) - 20} more (full list saved to JSON below)")
    print()


The previous cell only previewed the first 20 per dataset. Because there are tens of thousands of scenes, printing them all would be very long, so the cell below writes the **full list** to a JSON file you can open at any time or `import` from other scripts.


In [ ]:
import json

out_path = os.path.join(PROJ, "output", "scene_inventory.json")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(inventory, f, ensure_ascii=False, indent=1)
print(f"Full scene inventory saved: {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1024:.1f} KB")
print()
for name in DATASETS:
    print(f"  {name}: {len(inventory[name])} scenes")
print(f"  total: {sum(len(v) for v in inventory.values())} scenes")


## Summary

| step | command | ran locally | output |
|---|---|---|---|
| 1. GLB → USD (single) | `glb_to_usd.py` | ✅ | `output/normalized/scannet_scene0001_00/scene.usd` |
| 2. GLB → USD (batch) | `usd_all.sh` | ✅ | `output/normalized/<ds>_<id>/scene.usd` × 5 |
| 3. Blender render | `glb_render.py` | ✅ | `output/render/scannet/scene0001_00/perspective.png` |
| 4. top-down projection | `topdown_projection.py` | ✅ | `output/topdown/scannet_scene0001_00_topdown.png` |

For the full batch (render + top-down + scene-info JSON for a random sample of 50 scenes per category), see `scripts/batch_pipeline.py` and the README.
